# فصل ۱۱: تعبیه متن (Text Embeddings)
## بخش ۱: پیاده‌سازی عملی مدل‌های تعبیه متن و سنجش شباهت در LangChain

### مفاهیم کلیدی:
1. **بردار تعبیه (Embedding Vector):** تبدیل متن به یک بردار عددی چندبعدی (مثلاً ۷۶۸ یا ۱۵۳۶ بعد) که معنا و مفهوم معنایی (Semantic Meaning) متن را نمایش می‌دهد.
2. **شباهت کسینوسی (Cosine Similarity):** معیاری بین `-1` تا `1` (معمولاً در تعبیه‌ها بین `0` تا `1`) که زاویه بین دو بردار را می‌سنجد. هر چه به `1` نزدیک‌تر باشد، دو متن هم‌معنی‌ترند و شباهت یک متن با خودش دقیقاً `1.0` است.
3. **دو متد اصلی در LangChain:**
   - `embed_query(text)`: بردارسازی یک رشته متنی منفرد (مناسب برای پرسش و کوئری جستجو).
   - `embed_documents([doc1, doc2, ...])`: بردارسازی دسته‌ای لیستی از اسناد و متون.

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv

# حذف پروکسی‌های قدیمی و غیرفعال محیطی برای جلوگیری از خطای WinError 10061
for var in ["http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY", "all_proxy", "ALL_PROXY"]:
    os.environ.pop(var, None)

# بارگذاری متغیرهای محیطی
load_dotenv()

# تابع سنجش شباهت کسینوسی (سازگار با sklearn و دارای جایگزین بومی numpy)
try:
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError:
    def cosine_similarity(a, b):
        """محاسبه کسینوس شباهت با numpy بدون نیاز به کتابخانه sklearn"""
        a = np.asarray(a)
        b = np.asarray(b)
        dot_val = np.dot(a, b.T)
        norm_a = np.linalg.norm(a, axis=-1, keepdims=True)
        norm_b = np.linalg.norm(b, axis=-1, keepdims=True)
        return dot_val / (norm_a * norm_b.T)

print("✅ تنظیمات اولیه و تابع cosine_similarity بارگذاری شد!")

### ۱. مدل‌های تجاری و ابری (Cloud / API Models)

در پروژه‌ها می‌توانید بر حسب نیاز از مدل‌های گوگل جمینای یا مدل‌های OpenAI/تپ‌سیج استفاده کنید.
در اینجا مدل **`GoogleGenerativeAIEmbeddings`** (مدل `text-embedding-004`) با کلید `.env` فعلی شما آماده کار است، و کد معادل OpenAI نیز قرار داده شده است:

In [ ]:
# ۱. استفاده از مدل امبدینگ گوگل جمینای (پیش‌فرض آماده در این دوره)
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
print("✅ مدل تعبیه متن جمینای (text-embedding-004) با موفقیت متصل شد!")

# ۲. در صورت تمایل به استفاده از OpenAI / تپ‌سیج مطابق درسنامه:
# from langchain_openai import OpenAIEmbeddings
# embeddings_model = OpenAIEmbeddings(
#     model="text-embedding-3-small",
#     base_url="https://api.tapsage.com/openai/v1"
# )

### ۲. استخراج بردار تعبیه با متد `embed_query`

سه جمله زیر را بررسی می‌کنیم:
- `text1`: «فوتبال بهترین ورزش است.»
- `text2`: «برترین ورزش فوتبال است.» (کاملاً هم‌معنی با جمله اول اما با کلمات متفاوت)
- `text3`: «هوا بسیار سرد شده است.» (موضوعی کاملاً بی‌ربط)

In [ ]:
text1 = "فوتبال بهترین ورزش است."
text2 = "برترین ورزش فوتبال است."
text3 = "هوا بسیار سرد شده است."

# دریافت بردارهای تعبیه به صورت تک‌متن
embed1 = embeddings_model.embed_query(text1)
embed2 = embeddings_model.embed_query(text2)
embed3 = embeddings_model.embed_query(text3)

print(f"تعداد ابعاد بردار تعبیه: {len(embed1)}")
print(f"نمونه ۱۰ عدد اول بردار: {embed1[:10]}")

### ۳. محاسبه و سنجش شباهت کسینوسی (Cosine Similarity)

با تبدیل لیست‌ها به آرایه‌های دو بعدی (`reshape(1, -1)`)، شباهت معنایی جملات را می‌سنجیم:

In [ ]:
embed1_array = np.array(embed1).reshape(1, -1)
embed2_array = np.array(embed2).reshape(1, -1)
embed3_array = np.array(embed3).reshape(1, -1)

similarity12 = cosine_similarity(embed1_array, embed2_array)[0][0]
similarity13 = cosine_similarity(embed1_array, embed3_array)[0][0]
similarity23 = cosine_similarity(embed2_array, embed3_array)[0][0]
similarity11 = cosine_similarity(embed1_array, embed1_array)[0][0]

print(f"Cosine Similarity 1 & 2 (فوتبال و فوتبال): {similarity12:.4f}")
print(f"Cosine Similarity 1 & 3 (فوتبال و هوا):     {similarity13:.4f}")
print(f"Cosine Similarity 2 & 3 (ورزش و هوا):       {similarity23:.4f}")
print(f"Cosine Similarity 1 & 1 (شباهت با خود):    {similarity11:.4f}")

### ۴. بردارسازی دسته‌ای با `embed_documents`

زمانی که تعداد زیادی متن یا سند داریم، به جای حلقه و فراخوانی مکرر `embed_query`، از متد **`embed_documents`** استفاده می‌کنیم تا در یک درخواست دسته‌ای (Batch) با سرعت بالا همه را بردارسازی کند:

In [ ]:
texts = [text1, text2, text3]
embeddings = embeddings_model.embed_documents(texts)

print(f"تعداد متون پردازش‌شده: {len(embeddings)}")
print(f"طول بعد هر بردار: {len(embeddings[0])}")
print(f"نمونه اعداد بردار اول: {embeddings[0][:5]}")

### ۵. مدل‌های متن‌باز و لوکال با Ollama (`bge-m3`)

اگر بخواهید بدون وابستگی به اینترنت یا هزینه API، تعبیه‌ها را روی سیستم خود بسازید، مدل چندزبانه **`bge-m3`** از طریق ابزار **Ollama** یکی از بهترین گزینه‌ها به خصوص برای زبان فارسی است.

#### مراحل استقرار مدل در ترمینال:
```bash
# دانلود مدل تعبیه متن bge-m3 در Ollama
ollama pull bge-m3
```

#### استفاده در پایتون با `langchain_ollama`:
```python
from langchain_ollama import OllamaEmbeddings

local_embeddings = OllamaEmbeddings(model="bge-m3")
text = "من کدنویسی را دوست دارم."
embed = local_embeddings.embed_query(text)
print(f"تعداد ابعاد بردار bge-m3: {len(embed)}")  # 1024 بعد
```

--- 

### جدول پارامترهای تنظیمی کلاس `OllamaEmbeddings`:

| نام پارامتر | نوع داده / مقدار پیش‌فرض | توضیحات و کاربرد |
| :--- | :--- | :--- |
| `model` | `str` (الزامی) | نام مدل دانلود شده در Ollama (مثلاً `"bge-m3"`). |
| `base_url` | `str` / `"http://localhost:11434"` | آدرس سرور Ollama. |
| `num_ctx` | `int` / `2048` | طول پنجره بافتار. مدل `bge-m3` تا `8192` توکن را پشتیبانی می‌کند. اگر متون شما طولانی است، این مقدار را مثلاً `8192` قرار دهید. |
| `num_gpu` | `int` / `-1` | تعداد لایه‌هایی که روی GPU لود می‌شوند (`-1` بیشترین مقدار ممکن را به صورت خودکار انتخاب می‌کند). |
| `num_thread` | `int` / خودکار | تعداد هسته‌های CPU که در صورت عدم استفاده از کارت گرافیک فعال می‌شوند. |
| `keep_alive` | `str` / `"5m"` | مدت زمانی که مدل پس از آخرین درخواست در RAM/VRAM باقی می‌ماند (مثلاً `"1h"` مانع لود و آنلود مکرر می‌شود). |
| `show_progress`| `bool` / `False` | نمایش نوار پیشرفت هنگام پردازش‌های سنگین. |